# coffee-leaf-ai, Phase 2: healthy / rust / other disease

Data: private Kaggle dataset made from `data/phase2_kaggle.zip` (`ml/scripts/08_build_phase2_dataset.py`).
Training code: `ml/src/phase2.py` from the dataset, checksum-verified below.

**Order of work (each step only uses what the rules allow)**
1. Integrity checks (fail = stop).
2. Leave-one-dataset-out: train without one dataset, test on it. Estimates performance on a new kind of photo.
3. Final model: 3 seeds on all training data; best epoch and deployed seed chosen on validation
   (worst per-dataset balanced accuracy).
4. Calibration (temperature in [0.5, 5]) and the "not sure" rule (confidence threshold + distance to the training
   data), all fitted on validation.
5. One evaluation on `test_locked` (RoCoLe farms never used before) and on `ood`, against criteria fixed in advance.
6. ONNX export with logits + features, checked against PyTorch on real images (mismatch = stop).

In [ ]:
%pip install -q timm onnx onnxscript onnxruntime

In [ ]:
import os, sys, glob, json, hashlib, gc
from pathlib import Path

def find_project_root():
    if os.environ.get("COFFEE_ROOT"):
        return Path(os.environ["COFFEE_ROOT"])
    hits = glob.glob("/kaggle/input/**/data/processed_phase2/manifest.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("Attach the Phase 2 dataset (made from data/phase2_kaggle.zip).")
    return Path(hits[0]).parents[2]

ROOT = find_project_root()
OUT = Path(os.environ.get("COFFEE_OUT") or ("/kaggle/working/phase2" if Path("/kaggle/working").exists() else ROOT / "artifacts" / "phase2"))
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "ml"))

EXPECTED_SHA256 = {
    "common.py": "6ad5c06f2fc35db20d4054ebc97e3950d0fcc1390c7b31a092404fc626c676cf",
    "crops.py": "40fd2e3501a04f6c158a0f232e73b2d1e97c1886cfd1b5ae19d1087b3e80f79e",
    "evaluate.py": "5d4fb317669c7e64cd729e18550816812214266767e5422fe44c858fa182bca7",
    "export_onnx.py": "b1b761c4b41b1092779af9db54e0d0f4e6d32dc3ec027678659dca6293c90760",
    "phase2.py": "d7c01ea06851ddb137c3efb80d38b9e9d8a6c3dfe17f322c071ad899f67f29a5"
}
for f, h in EXPECTED_SHA256.items():
    got = hashlib.sha256(Path(ROOT, "ml", "src", f).read_bytes()).hexdigest()
    assert got == h, f"ml/src/{f} in the dataset differs from the reviewed version"
print("project root:", ROOT, "| outputs:", OUT, "| code checksums OK")

In [ ]:
import numpy as np, pandas as pd, torch
from src import phase2 as P

base, manifest, crops = P.load_tables(ROOT)
CLASSES = P.CLASSES
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

FIT = ["train", "val"]
assert manifest.loc[manifest.split.eq("test_locked"), "source"].eq("rocole").all(), "test_locked must be RoCoLe only"
locked_farms = set(manifest.loc[manifest.split.eq("test_locked"), "farm_id"])
assert not (set(manifest.loc[manifest.split.isin(FIT + ["test"]), "farm_id"]) - {""}) & locked_farms, "locked farm leaked"
held_groups = set(manifest.loc[manifest.split.isin(["test_locked", "ood"]), "group"])
assert not set(manifest.loc[manifest.split.isin(FIT + ["test"]), "group"]) & held_groups, "group shared with held-out data"
assert manifest.loc[manifest.split.isin(FIT), "ood_type"].eq("").all(), "OOD rows in train/val"
assert set(manifest.loc[manifest.split.isin(FIT), "task_label"]) == set(CLASSES)
print(manifest.groupby(["split", "source", "task_label"]).size().unstack(fill_value=0))
print("locked farms:", sorted(locked_farms))

In [ ]:
HP = dict(
    model="convnext_tiny.fb_in22k_ft_in1k", pretrained=True, size=224, resize=256,
    batch_size=32, workers=4, epochs=15, patience=4, epoch_size=4000,
    lr=1e-4, weight_decay=0.05, label_smoothing=0.1, ema_decay=0.998, share_tolerance=0.005,
    seeds=[0, 1, 2],
    knn_k=10, ood_val_quantile=0.95,     # "not sure" if farther from the training data than 95% of val images
    target_accuracy=0.95,                # confidence threshold: >= 95% val accuracy on answered images
    criteria=dict(locked_macro_f1=0.90, answered_accuracy=0.95, answered_share=0.70,
                  other_as_rust=0.05, ood_rejected=0.80, val_ece=0.05),   # fixed before any result is seen
)
LODO = {"bracol": ["bracol"], "uganda": ["uganda"], "jmuben": ["jmuben", "jmuben2"], "rocole": ["rocole"]}

if os.environ.get("COFFEE_SMOKE") == "1":  # tiny CPU run to check the notebook end to end
    HP.update(pretrained=False, model="convnext_atto", epochs=1, epoch_size=64, batch_size=8, workers=0,
              seeds=[0, 1], share_tolerance=1.0)
    manifest = pd.concat([g.sample(frac=1, random_state=0).groupby(["source", "task_label", "ood_type"]).head(5)
                          for _, g in manifest.groupby("split")])
    LODO = {"bracol": ["bracol"]}
print(HP)

train_all = manifest[manifest.split.eq("train")].reset_index(drop=True)
val_all = manifest[manifest.split.eq("val")].reset_index(drop=True)

## 2. Leave-one-dataset-out (estimate for a new kind of photo)

In [ ]:
_, tf_eval = P.transforms(HP["size"], HP["resize"])
lodo = {}
for held, srcs in LODO.items():
    tr = train_all[~train_all.source.isin(srcs)].reset_index(drop=True)
    va = val_all[~val_all.source.isin(srcs)].reset_index(drop=True)
    model, hist, best, shares = P.train_model(f"lodo-{held}", tr, va, base, crops, CLASSES, HP, DEVICE, seed=0)
    ev = manifest[manifest.split.isin(["val", "test"]) & manifest.source.isin(srcs) & manifest.is_group_rep]
    y, z, _ = P.predict(model, P.eval_loader(base, crops, CLASSES, HP, ev, tf=tf_eval), DEVICE)
    mt = P.compute_metrics(y, P.softmax(z), CLASSES)
    lodo[held] = {"n": mt["n"], "accuracy": mt["accuracy"], "macro_f1": mt["macro_f1"], "best_epoch": best["epoch"],
                  "per_class_f1": {k: v["f1-score"] for k, v in mt["per_class"].items()},
                  "confusion_matrix": mt["confusion_matrix"]}
    print(f"LODO held-out {held}: n={mt['n']} acc={mt['accuracy']:.3f} macro_f1={mt['macro_f1']:.3f}")
    del model; gc.collect(); torch.cuda.empty_cache()
(OUT / "lodo.json").write_text(json.dumps(lodo, indent=2))
pd.DataFrame(lodo).T[["n", "accuracy", "macro_f1", "best_epoch"]]

## 3. Final model: 3 seeds, deployed seed chosen on validation

In [ ]:
seed_runs = {}
for seed in HP["seeds"]:
    model, hist, best, shares = P.train_model(f"final-s{seed}", train_all, val_all, base, crops, CLASSES, HP, DEVICE, seed)
    seed_runs[seed] = {"state": {k: v.cpu() for k, v in model.state_dict().items()}, "history": hist, "best": best}
    del model; gc.collect(); torch.cuda.empty_cache()
deployed_seed = max(seed_runs, key=lambda s: seed_runs[s]["best"]["score"])
print({s: round(r["best"]["score"], 4) for s, r in seed_runs.items()}, "-> deployed seed", deployed_seed)
print("training sampling shares:\n", shares.round(4))

In [ ]:
# Per-seed in-domain test scores (to show stability; not used for any choice)
test_rep = manifest[manifest.split.eq("test") & manifest.is_group_rep].reset_index(drop=True)
seed_table = []
for seed, r in seed_runs.items():
    m_ = P.Net(HP["model"], False, len(CLASSES)); m_.load_state_dict(r["state"]); m_ = m_.to(DEVICE).eval()
    y, z, _ = P.predict(m_, P.eval_loader(base, crops, CLASSES, HP, test_rep, tf=tf_eval), DEVICE)
    mt = P.compute_metrics(y, P.softmax(z), CLASSES)
    seed_table.append({"seed": seed, "val_worst_source_bal_acc": r["best"]["score"], "best_epoch": r["best"]["epoch"],
                       "test_acc": mt["accuracy"], "test_macro_f1": mt["macro_f1"]})
    del m_; gc.collect(); torch.cuda.empty_cache()
seed_table = pd.DataFrame(seed_table).set_index("seed")
print(seed_table.round(4)); print("mean:", seed_table.mean().round(4).to_dict(), "| std:", seed_table.std().round(4).to_dict())

## 4-5. Calibration, 'not sure' rule (validation only), then one look at the held-out sets

In [ ]:
model = P.Net(HP["model"], False, len(CLASSES)); model.load_state_dict(seed_runs[deployed_seed]["state"])
model = model.to(DEVICE).eval()

def run(df, view="full"):
    return P.predict(model, P.eval_loader(base, crops, CLASSES, HP, df.reset_index(drop=True), view=view, tf=tf_eval), DEVICE)

bank_df = train_all[train_all.is_group_rep]
_, _, bank = run(bank_df)
val_rep = val_all[val_all.is_group_rep].reset_index(drop=True)
yv, zv, fv = run(val_rep)
T = P.fit_temperature_bounded(zv, yv)
pv = P.softmax(zv / T)
ood_v = P.knn_ood_score(fv, bank, HP["knn_k"])
TH = P.choose_thresholds(yv, pv, ood_v, HP)
print(f"temperature {T:.3f} | thresholds {TH} (all from validation)")

results = {"val": P.metrics_block(yv, pv, CLASSES, val_rep.source.to_numpy())}
results["val"]["selective"] = P.selective_report(yv, pv, ood_v, TH, CLASSES)
for name, df, view in [("test", test_rep, "full"),
                       ("test_locked", manifest[manifest.split.eq("test_locked")].reset_index(drop=True), "full"),
                       ("test_locked_leafbox", manifest[manifest.split.eq("test_locked")].reset_index(drop=True), "leaf")]:
    y, z, f = run(df, view)
    p, o = P.softmax(z / T), P.knn_ood_score(f, bank, HP["knn_k"])
    results[name] = P.metrics_block(y, p, CLASSES, df.source.to_numpy())
    results[name]["selective"] = P.selective_report(y, p, o, TH, CLASSES)
    P.plot_confusion_matrix(results[name]["confusion_matrix"], CLASSES, OUT / f"cm_{name}.png", title=name)
    P.plot_reliability(y, p, OUT / f"reliability_{name}.png", title=name)

ood_df = manifest[manifest.split.eq("ood")].reset_index(drop=True)
_, zo, fo = run(ood_df)
po, oo = P.softmax(zo / T), P.knn_ood_score(fo, bank, HP["knn_k"])
ans_o = P.decide(po, oo, TH)
results["ood"] = {"n": int(len(ood_df)), "rejected_share": float((~ans_o).mean()),
                  "answered_as": {c: float((ans_o & (po.argmax(1) == i)).mean()) for i, c in enumerate(CLASSES)},
                  "by_group": {g: float((~ans_o[idx]).mean()) for g, idx in ood_df.groupby(ood_df.folder).indices.items()}}
from sklearn.metrics import roc_auc_score
_, zt, ft = run(test_rep)
id_ood, id_conf = P.knn_ood_score(ft, bank, HP["knn_k"]), P.softmax(zt / T).max(1)
lab = np.r_[np.zeros(len(id_ood)), np.ones(len(oo))]
results["ood"]["auroc_knn_distance"] = float(roc_auc_score(lab, np.r_[id_ood, oo]))
results["ood"]["auroc_max_probability"] = float(roc_auc_score(lab, -np.r_[id_conf, po.max(1)]))

crit = P.criteria_table(results, HP)
print(crit.to_string(index=False))

## 6. Export and parity check

In [ ]:
exp = P.Exported(model.cpu(), T)
parity_df = pd.concat([test_rep, manifest[manifest.split.eq("test_locked")]]).head(256)
parity = P.export_and_verify(exp, OUT / "model.onnx", HP["size"], P.eval_loader(base, crops, CLASSES, HP, parity_df, tf=tf_eval))
np.savez_compressed(OUT / "ood_bank.npz", features=bank.astype(np.float16))
(OUT / "labels.json").write_text(json.dumps(CLASSES))
(OUT / "preprocess.json").write_text(json.dumps({
    "resize_shorter": HP["resize"], "crop_size": HP["size"], "interpolation": "bilinear",
    "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225], "color": "RGB", "layout": "NCHW", "scale": "0-1",
    "output": "logits (temperature-scaled), features (L2-normalised)", "temperature": T}, indent=2))
(OUT / "decision.json").write_text(json.dumps({
    "confidence_threshold": TH["confidence"], "ood_threshold": TH["ood"], "knn_k": HP["knn_k"],
    "rule": "answer only if max probability >= confidence_threshold and kNN distance <= ood_threshold",
    "fitted_on": "validation split only"}, indent=2))
report = {"hp": HP, "deployed_seed": deployed_seed, "temperature": T, "thresholds": TH, "lodo": lodo,
          "seed_table": seed_table.reset_index().to_dict("records"), "results": results, "parity": parity,
          "criteria": crit.to_dict("records"),
          "histories": {s: r["history"] for s, r in seed_runs.items()}}
(OUT / "metrics.json").write_text(json.dumps(report, indent=2, default=float))
print("parity:", parity)
print(json.dumps({k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items() if kk in ("n", "accuracy", "macro_f1", "ece", "selective")}
                  for k, v in results.items() if k != "ood"}, indent=1, default=float))
print("OOD:", json.dumps(results["ood"], indent=1, default=float))